In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
WORK = '/kaggle/working'

def find(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    assert len(hits) == 1, (name, hits)
    return hits[0]

BASE = os.path.dirname(find('train.csv'))
test = pd.read_csv(f'{BASE}/test.csv')
folds = pd.read_csv(find('folds.csv'))
te_uids = ['test/' + f for f in test.filename]
dur_te = pd.read_csv(find('audio_meta.csv')).set_index('uid').duration.reindex(te_uids).values
te_grp = np.where(dur_te < 40, 'short', np.where(dur_te < 52, 'dur_45s', 'dur_60s'))

TF = pd.read_csv(find('text_features.csv'), index_col='uid')
LS = pd.read_csv(find('llm_scores.csv')).set_index('uid')
V = pd.read_csv(find('llm_views.csv')).set_index('uid')
Q = pd.read_csv(find('llm14_scores.csv')).set_index('uid')
llm_cols = ['e', 'ent'] + [f'p{k}' for k in range(1, 6)] + [f'l{k}' for k in range(1, 6)]
VIEWS = ('rubric', 'errors', 'wellformed', 'complexity')
v_cols = [f'{v}_{s}' for v in VIEWS for s in ['e', 'ent', 'p1', 'p2', 'p3', 'p4', 'p5']]
q_cols = ['e', 'ent', 'p1', 'p2', 'p3', 'p4', 'p5']
cols = TF.columns.tolist()
conf = [c for c in cols if c.startswith(('wp_', 'seg_logprob', 'max_no_speech'))]
noconf = [c for c in cols if c not in conf]
tu = folds.uid
XT = {'TL2': np.hstack([TF.loc[tu, cols].values, LS.loc[tu, llm_cols].values, V.loc[tu, v_cols].values]),
      'TL14': np.hstack([TF.loc[tu, cols].values, LS.loc[tu, llm_cols].values, Q.loc[tu, q_cols].values]),
      'TLn14': np.hstack([TF.loc[tu, noconf].values, LS.loc[tu, llm_cols].values, Q.loc[tu, q_cols].values])}

def load_emb(emb_file, uid_file):
    E = np.load(find(emb_file), mmap_mode='r')
    ids = pd.read_csv(find(uid_file)).uid.tolist()
    ix = {u: i for i, u in enumerate(ids)}
    return E, np.array([ix[u] for u in folds.uid]), np.array([ix[u] for u in te_uids])

EW, w_tr, w_te = load_emb('wavlm_large_mean.npy', 'wavlm_uids.csv')
ES = np.load(find('wavlm_large_std.npy'), mmap_mode='r')
EH, h_tr, _ = load_emb('whisper_enc_mean.npy', 'whisper_enc_uids.csv')
E7, s_tr, _ = load_emb('llm_rubric_meanpool.npy', 'llm_views_uids.csv')
E14, q_tr, _ = load_emb('llm14_meanpool.npy', 'llm14_uids.csv')
POOL7, POOL14 = [6, 9, 12, 16, 20, 24], [10, 15, 21, 27, 34, 41]
j7, j14 = POOL7.index(16), POOL14.index(21)
XE = {'W': np.asarray(EW[w_tr, 21], dtype=np.float32),
      'H': np.asarray(EH[h_tr, 30], dtype=np.float32),
      'M7': np.asarray(E7[s_tr, j7], dtype=np.float32),
      'M14': np.asarray(E14[q_tr, j14], dtype=np.float32)}
X4_tr = np.asarray(EW[w_tr, 4], dtype=np.float32)
X4_te = np.asarray(EW[w_te, 4], dtype=np.float32)
print({k: v.shape for k, v in XE.items()})
print({k: v.shape for k, v in XT.items()})

y = folds.label.values
w = folds.w_test.values
nz = (folds.is_zero == 0).values
grp = folds.dur_group.values
idx_nz = np.where(nz)[0]

def rmse(a, b): return float(np.sqrt(np.mean((a - b) ** 2)))
def pear(a, b): return float(np.corrcoef(a, b)[0, 1])
def wrmse(a, b, ww): return float(np.sqrt(np.sum(ww * (a - b) ** 2) / np.sum(ww)))
def wpear(a, b, ww):
    ma, mb = np.average(a, weights=ww), np.average(b, weights=ww)
    return float(np.sum(ww * (a - ma) * (b - mb)) /
                 np.sqrt(np.sum(ww * (a - ma) ** 2) * np.sum(ww * (b - mb) ** 2)))

def score(oof):
    out = {}
    for g in ['dur_45s', 'dur_60s']:
        m = nz & (grp == g)
        out['pear_' + g[4:6]] = pear(y[m], oof[m])
    rm, pe = wrmse(y[nz], oof[nz], w[nz]), wpear(y[nz], oof[nz], w[nz])
    out.update(rmse_w=rm, pear_w=pe, proxy=(rm + 1 - pe) / 2)
    return out

class Winsor(BaseEstimator, TransformerMixin):
    def __init__(self, lo=1, hi=99):
        self.lo, self.hi = lo, hi
    def fit(self, X, y=None):
        self.lo_ = np.percentile(X, self.lo, axis=0)
        self.hi_ = np.percentile(X, self.hi, axis=0)
        return self
    def transform(self, X):
        return np.clip(X, self.lo_, self.hi_)

def ridge_text():
    return make_pipeline(SimpleImputer(strategy='median'), Winsor(), StandardScaler(),
                         RidgeCV(alphas=np.logspace(-1, 4, 12)))
def ridge_hid():
    return make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(1, 6, 11)))
def ridge_wavlm():
    return make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(1, 5, 9)))
def svr_head():
    return make_pipeline(StandardScaler(), PCA(n_components=128, random_state=0),
                         SVR(C=3.0, epsilon=0.2, gamma='scale'))

In [ ]:
def l2n(Z):
    return Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-9)

ybin = np.digitize(y, [2.25, 3.25, 4.25])
strat = ybin * 2 + (grp == 'dur_60s').astype(int)

def make_folds(layers, k, seed):
    mean_ = np.asarray(EW[w_tr][:, layers, :], dtype=np.float32).reshape(len(y), -1)
    std_ = np.asarray(ES[w_tr][:, layers, :], dtype=np.float32).reshape(len(y), -1)
    G = np.hstack([mean_, std_])
    sc = StandardScaler().fit(G[nz])
    Z = PCA(n_components=50, random_state=seed).fit(sc.transform(G[nz])).transform(sc.transform(G))
    cl = AgglomerativeClustering(n_clusters=k, metric='cosine', linkage='average').fit_predict(Z[nz])
    fold = np.random.RandomState(seed).randint(0, 5, len(y))
    sg = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    for f, (_, va) in enumerate(sg.split(idx_nz, strat[nz], groups=cl)):
        fold[idx_nz[va]] = f
    return fold, cl

def nn_sim(fold):
    val_sim = np.full(len(y), np.nan)
    te_sim = np.zeros((5, len(X4_te)))
    for f in range(5):
        va = fold == f
        ref, qry = (~va) & nz, va & nz
        sc = StandardScaler().fit(X4_tr[ref])
        Rm, Qm, Tm = (l2n(sc.transform(a)) for a in (X4_tr[ref], X4_tr[qry], X4_te))
        val_sim[qry] = (Qm @ Rm.T).max(1)
        te_sim[f] = (Tm @ Rm.T).max(1)
    return val_sim, te_sim.mean(0)

SCHEMES = {'A': ((0, 2, 4), 180, 7), 'B': ((1, 3, 5), 90, 11), 'C': ((2, 4, 6), 250, 23)}
FOLDS, rows = {}, []
for name, (layers, k, seed) in SCHEMES.items():
    fold, cl = make_folds(layers, k, seed)
    FOLDS[name] = fold
    fs = np.bincount(fold[nz], minlength=5)
    vs, ts = nn_sim(fold)
    row = dict(scheme=name, layers=str(layers), k=k, fold_min=int(fs.min()), fold_max=int(fs.max()),
               biggest_cluster=int(np.bincount(cl).max()))
    for g in ['dur_45s', 'dur_60s']:
        row[f'sim_val_{g[4:6]}'] = vs[nz & (grp == g)].mean()
        row[f'sim_test_{g[4:6]}'] = ts[te_grp == g].mean()
    rows.append(row)
print(pd.DataFrame(rows).round(3).to_string(index=False))

In [ ]:
def oof_for(X, fold, make_model):
    oof = np.full(len(y), np.nan)
    for f in range(5):
        va = fold == f
        m = make_model().fit(X[(~va) & nz], y[(~va) & nz])
        oof[va & nz] = np.clip(m.predict(X[va & nz]), 1, 5)
    return oof

MK = {'W': ridge_wavlm, 'H': ridge_hid, 'M7': ridge_hid, 'M14': ridge_hid}
OOF = {}
for sname, fold in FOLDS.items():
    o = {}
    for key in ['W', 'H', 'M7', 'M14']:
        o[key + '_r'] = oof_for(XE[key], fold, MK[key])
        o[key + '_s'] = oof_for(XE[key], fold, svr_head)
    for key in ['TL2', 'TL14', 'TLn14']:
        o[key] = oof_for(XT[key], fold, ridge_text)
    OOF[sname] = o
    print('scheme', sname, 'done | whisper ridge proxy:', round(score(o['H_r'])['proxy'], 4))

In [ ]:
def mix(o, k): return (o[k + '_r'] + o[k + '_s']) / 2

CAND = {
    'whisper ridge alone': lambda o: o['H_r'],
    'Stack v4 pick: avg(W,H,TL2,M7) ridge only': lambda o: np.mean([o['W_r'], o['H_r'], o['TL2'], o['M7_r']], axis=0),
    'Stack v5 pick: avg(mix W,H,M7,M14 + TL14)': lambda o: np.mean([mix(o, 'W'), mix(o, 'H'), o['TL14'], mix(o, 'M7'), mix(o, 'M14')], axis=0),
    'Stack v5 fairness variant (TLn14)': lambda o: np.mean([mix(o, 'W'), mix(o, 'H'), o['TLn14'], mix(o, 'M7'), mix(o, 'M14')], axis=0),
    'Stack v5 without text models': lambda o: np.mean([mix(o, 'W'), mix(o, 'H'), mix(o, 'M7'), mix(o, 'M14')], axis=0),
    'Stack v5 ridge only (no SVR)': lambda o: np.mean([o['W_r'], o['H_r'], o['TL14'], o['M7_r'], o['M14_r']], axis=0),
}
sc = {c: {s: score(fn(OOF[s])) for s in OOF} for c, fn in CAND.items()}
rows = []
for c in CAND:
    r = {'candidate': c}
    for s in OOF:
        r[f'proxy_{s}'] = sc[c][s]['proxy']
    for k in ['proxy', 'rmse_w', 'pear_w', 'pear_45', 'pear_60']:
        r[k + ('_mean' if k == 'proxy' else '')] = np.mean([sc[c][s][k] for s in OOF])
    rows.append(r)
T = pd.DataFrame(rows).round(4).sort_values('proxy_mean')
print(T.to_string(index=False))

ref = 'Stack v4 pick: avg(W,H,TL2,M7) ridge only'
print('\nproxy difference against the Stack v4 pick, per scheme (negative = better):')
for c in CAND:
    if c == ref:
        continue
    d = [sc[c][s]['proxy'] - sc[ref][s]['proxy'] for s in OOF]
    print(f'  {c:45s} {np.round(d, 4)} | mean {np.mean(d):+.4f} | wins {int(sum(x < 0 for x in d))}/{len(d)}')

fig, ax = plt.subplots(figsize=(10, 4.5))
order = T.sort_values('proxy_mean', ascending=False)
ax.barh(order.candidate, order.proxy_mean, alpha=0.45)
for s in OOF:
    ax.scatter(order[f'proxy_{s}'], order.candidate, s=28, label=f'scheme {s}')
pc = [f'proxy_{s}' for s in OOF]
ax.set_xlim(order[pc].min().min() - 0.02, order[pc].max().max() + 0.01)
ax.set_title('Proxy on three fresh group-fold schemes (lower is better, axis truncated)')
ax.legend()
plt.tight_layout(); plt.show()
T.to_csv(f'{WORK}/fresh_fold_validation.csv', index=False)